# 04 — Stage-Specific Derived Features (Tier 1)

**8 new features**, all derived from existing HLS + Daymet data (no new downloads).

| Feature | Source | Window | Stage relevance |
|---|---|---|---|
| `NDVI_pre_dormancy_peak` | HLS NDVI | DOS 100-150 | emergence/tillering |
| `dormancy_break_DOS` | HLS NDVI | DOS 200-260 | tillering/jointing |
| `senescence_rate_post_POS` | HLS NDVI | DOS POS+1 → POS+60 | maturity |
| `days_above_5C_winter` | Daymet | DOS 150-220 | tillering |
| `frost_days_pre_jointing` | Daymet | DOS 200-260 | jointing |
| `heat_days_post_anthesis` | Daymet | DOS 295-360 | maturity |
| `vpd_cum_post_anthesis` | Daymet | DOS 295-360 | maturity |
| `frost_events_spring` | Daymet | DOS 240-280 | jointing/flag leaf |

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

FEAT_IN  = 'data/processed/features/features_with_solar_vpd.parquet'
FEAT_OUT = 'data/processed/features/features_stage_specific.parquet'
WX_PATH  = 'data/raw/satellite/daymet_daily_weather_full.csv'
RAW_PATH = 'data/processed/buffer/hls_phenology_merged.parquet'
MAP_PATH = 'data/processed/csb_to_buf_mapping.csv'

feat = pd.read_parquet(FEAT_IN)
raw  = pd.read_parquet(RAW_PATH)
wx   = pd.read_csv(WX_PATH)
wx['date'] = pd.to_datetime(wx['date'], format='mixed')
wx['year'] = wx['date'].dt.year
wx['doy'] = wx['date'].dt.dayofyear
wx['FIELDID'] = wx['FIELDID'].astype(str)
if 'tmin' in wx.columns: wx = wx.rename(columns={'tmin':'Tmin','tmax':'Tmax'})
wx['T_mean'] = (wx['Tmin'] + wx['Tmax']) / 2
wx['month'] = wx['date'].dt.month
wx['gs_start_year'] = np.where(wx['month'] >= 7, wx['year'], wx['year'] - 1)
wx['harvest_year'] = wx['gs_start_year'] + 1
wx['gs_start_date'] = pd.to_datetime(wx['gs_start_year'].astype(str) + '-07-01')
wx['dos'] = (wx['date'] - wx['gs_start_date']).dt.days + 1

mapping = {}
if os.path.exists(MAP_PATH):
    m_df = pd.read_csv(MAP_PATH)
    mapping = dict(zip(m_df['field_id'].astype(str), m_df['nearest_BUF'].astype(str)))

print(f'Features: {feat.shape}, Raw HLS: {raw.shape}, Weather: {wx.shape}')

## 1. NDVI-derived features (from raw HLS time series)

In [ ]:
from scipy.signal import savgol_filter

# Compute DOS for raw HLS observations
raw['date'] = pd.to_datetime(raw['date'])
raw['month'] = raw['date'].dt.month
raw['cy'] = raw['date'].dt.year
raw['gs_start_year'] = np.where(raw['month'] >= 7, raw['cy'], raw['cy'] - 1)
raw['harvest_year'] = raw['gs_start_year'] + 1
raw['gs_start_date'] = pd.to_datetime(raw['gs_start_year'].astype(str) + '-07-01')
raw['dos'] = (raw['date'] - raw['gs_start_date']).dt.days + 1

def ndvi_features_per_field_year(fid, harvest_year):
    """Extract 3 NDVI-derived features per field-year."""
    sub = raw[(raw['field_id']==fid) & (raw['harvest_year']==harvest_year) & raw['NDVI'].notna()]
    if len(sub) < 5:
        return {'NDVI_pre_dormancy_peak': np.nan,
                'dormancy_break_DOS':     np.nan,
                'senescence_rate_post_POS': np.nan}
    sub = sub.sort_values('dos')
    
    # Smooth NDVI on DOS grid
    target_dos = np.arange(1, 366)
    daily = np.interp(target_dos, sub['dos'].values, sub['NDVI'].values)
    smoothed = savgol_filter(daily, min(15, len(daily)//2*2-1) if len(daily)>=5 else 3, 2) \
               if len(daily) >= 5 else daily
    
    out = {}
    
    # 1. Pre-dormancy NDVI peak (DOS 100-150 = Oct-Nov)
    pre_d = smoothed[99:150]  # DOS 100-150 (0-indexed: 99-149)
    out['NDVI_pre_dormancy_peak'] = float(pre_d.max()) if len(pre_d) > 0 else np.nan
    
    # 2. Dormancy break DOS — first DOS (200-260) where NDVI rises >5% from min
    win = smoothed[199:260]
    if len(win) > 5:
        baseline = win.min()
        threshold = baseline + 0.05  # 5% above min
        risen = np.where(win > threshold)[0]
        out['dormancy_break_DOS'] = int(risen[0] + 200) if len(risen) > 0 else np.nan
    else:
        out['dormancy_break_DOS'] = np.nan
    
    # 3. Senescence rate — slope of NDVI from POS to POS+60 days
    pos_idx = int(np.argmax(smoothed[239:359]) + 240) if len(smoothed) > 240 else np.nan
    if not np.isnan(pos_idx):
        end_idx = min(pos_idx + 60, 365)
        if end_idx > pos_idx + 5:
            seg = smoothed[pos_idx-1:end_idx]
            if len(seg) > 5:
                slope = (seg[-1] - seg[0]) / len(seg)  # NDVI/day (negative)
                out['senescence_rate_post_POS'] = float(slope)
            else:
                out['senescence_rate_post_POS'] = np.nan
        else:
            out['senescence_rate_post_POS'] = np.nan
    else:
        out['senescence_rate_post_POS'] = np.nan
    
    return out

ndvi_records = []
fy_pairs = feat[['field_id', 'year']].drop_duplicates().values
print(f'Computing NDVI features for {len(fy_pairs)} field-years...')
for i, (fid, yr) in enumerate(fy_pairs):
    out = ndvi_features_per_field_year(str(fid), int(yr))
    out['field_id'] = str(fid); out['year'] = int(yr)
    ndvi_records.append(out)
    if (i+1) % 500 == 0:
        print(f'  {i+1}/{len(fy_pairs)}')
ndvi_feat = pd.DataFrame(ndvi_records)
print(f'\nNDVI features computed: {ndvi_feat.shape}')
print(ndvi_feat.describe().round(3))

## 2. Weather-derived features (from Daymet)

In [ ]:
WX_FEATURES = {
    'days_above_5C_winter':   {'expr': lambda d: ((d['T_mean']>5) & (d['dos']>=150) & (d['dos']<=220)).sum()},
    'frost_days_pre_jointing':{'expr': lambda d: ((d['Tmin']<0) & (d['dos']>=200) & (d['dos']<=260)).sum()},
    'heat_days_post_anthesis':{'expr': lambda d: ((d['Tmax']>30) & (d['dos']>=295) & (d['dos']<=360)).sum()},
    'frost_events_spring':    {'expr': lambda d: ((d['Tmin']<-5) & (d['dos']>=240) & (d['dos']<=280)).sum()},
    'vpd_cum_post_anthesis':  {'expr': lambda d: d.loc[(d['dos']>=295)&(d['dos']<=360),'vpd'].sum()},
}

wx_records = []
print(f'Computing weather features for {len(fy_pairs)} field-years...')
for i, (fid, yr) in enumerate(fy_pairs):
    fid = str(fid); yr = int(yr)
    wx_fid = mapping.get(fid, fid)
    sub = wx[(wx['FIELDID']==wx_fid) & (wx['harvest_year']==yr)]
    if len(sub) == 0:
        wx_records.append({'field_id': fid, 'year': yr,
                          **{k: np.nan for k in WX_FEATURES}})
        continue
    row = {'field_id': fid, 'year': yr}
    for fname, spec in WX_FEATURES.items():
        try:
            row[fname] = float(spec['expr'](sub))
        except Exception:
            row[fname] = np.nan
    wx_records.append(row)
    if (i+1) % 500 == 0:
        print(f'  {i+1}/{len(fy_pairs)}')
wx_feat = pd.DataFrame(wx_records)
print(f'\nWeather features computed: {wx_feat.shape}')
print(wx_feat.describe().round(2))

## 3. Merge all + correlations with target

In [ ]:
ndvi_feat['field_id'] = ndvi_feat['field_id'].astype(str)
ndvi_feat['year'] = ndvi_feat['year'].astype(int)
wx_feat['field_id'] = wx_feat['field_id'].astype(str)
wx_feat['year'] = wx_feat['year'].astype(int)
feat['field_id'] = feat['field_id'].astype(str)
feat['year'] = feat['year'].astype(int)

feat_v4 = feat.merge(ndvi_feat, on=['field_id','year'], how='left') \
              .merge(wx_feat, on=['field_id','year'], how='left')

new_features = list(WX_FEATURES.keys()) + ['NDVI_pre_dormancy_peak','dormancy_break_DOS','senescence_rate_post_POS']
print('Correlations of new features with flag_true_doy:')
for c in new_features:
    sub = feat_v4[[c,'flag_true_doy']].dropna()
    if len(sub) < 50:
        print(f'  {c:32s}  n={len(sub):4d}  insufficient')
        continue
    r = np.corrcoef(sub[c], sub['flag_true_doy'])[0,1]
    print(f'  {c:32s}  n={len(sub):4d}  r={r:+.3f}')

feat_v4.to_parquet(FEAT_OUT, index=False)
print(f'\nSaved: {FEAT_OUT}')
print(f'Shape: {feat_v4.shape}')
print(f'New columns added: {new_features}')